# Relatório: O Desafio do Oráculo Preditivo - Mercado de Chicago
**Grupo 04:** Daniel Brito e João Vitor  
**Curso:** Desenvolvimento de Software Multiplataforma - FATEC Indaiatuba

---
## 1. Resumo
Este projeto apresenta a construção e comparação de modelos de Machine Learning para prever preços de imóveis em Chicago. Foram testadas três técnicas de regressão (Linear, Random Forest e SVR), sendo o modelo **Random Forest** o que apresentou melhor desempenho geral.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

# Carregando o dataset
df = pd.read_csv('Chicago-RE_HousePrice.csv')
print("Dados carregados com sucesso!")
df.head()

## 2. Análise Exploratória (EDA)
Realizamos uma análise visual para entender o comportamento dos preços em Chicago:
* **Distribuição:** O mercado é concentrado em casas de valores médios, com uma assimetria à direita (casas de luxo).
* **Correlação:** Vimos que `Space` (tamanho) é a variável que mais empurra o preço para cima.
* **Outliers:** Identificamos através do Boxplot que embora a maioria das casas esteja concentrada em uma faixa de preço específica, existem vários registros que chegam além do 90.

In [ ]:
# Criando os gráficos de análise
plt.figure(figsize=(15, 5))

# Boxplot
plt.subplot(1, 2, 1)
sns.boxplot(x=df['Price'], color='skyblue')
plt.title('Boxplot: Identificação de Outliers')

# Heatmap
plt.subplot(1, 2, 2)
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Heatmap: Correlação entre Variáveis')

plt.show()

## 3. Pré-processamento
Para garantir a qualidade dos modelos, realizamos:
1. **Tratamento de Nulos:** Preenchemos valores ausentes com a mediana.
2. **Escalonamento:** Padronizamos os dados usando `StandardScaler` para que variáveis em escalas diferentes (ex: nº de quartos vs impostos) não confundissem o modelo.
3. **Divisão:** Separamos 80% dos dados para treino e 20% para teste.

In [ ]:
# Limpeza e Split
imputer = SimpleImputer(strategy='median')
df_clean = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
X = df_clean.drop('Price', axis=1)
y = df_clean['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalonamento
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Modelos
models = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(kernel='rbf')
}

print(f"{'Modelo':<20} | {'R2':<6} | {'MAE':<6} | {'MAPE':<6}")
print("-" * 45)

for name, model in models.items():
    xt, xv = (X_train_sc, X_test_sc) if name != "Random Forest" else (X_train, X_test)
    model.fit(xt, y_train)
    p = model.predict(xv)
    
    r2 = r2_score(y_test, p)
    mae = mean_absolute_error(y_test, p)
    mape = mean_absolute_percentage_error(y_test, p) * 100
    
    print(f"{name:<20} | {r2:6.2f} | {mae:6.2f} | {mape:5.2f}%")

In [ ]:
# Limpeza e Split
imputer = SimpleImputer(strategy='median')
df_clean = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
X = df_clean.drop('Price', axis=1)
y = df_clean['Price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Escalonamento
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Modelos
models = {
    "Regressão Linear": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "SVR": SVR(kernel='rbf')
}

print(f"{'Modelo':<20} | {'R2':<6} | {'MAE':<6} | {'MAPE':<6} | {'RMSE':<6}")
print("-" * 60)

for name, model in models.items():
    xt, xv = (X_train_sc, X_test_sc) if name != "Random Forest" else (X_train, X_test)
    model.fit(xt, y_train)
    p = model.predict(xv)
    
    r2 = r2_score(y_test, p)
    mae = mean_absolute_error(y_test, p)
    mape = mean_absolute_percentage_error(y_test, p) * 100
    rmse = np.sqrt(mean_squared_error(y_test, p))
    
    print(f"{name:<20} | {r2:6.2f} | {mae:6.2f} | {mape:5.2f} | {rmse:6.2f}")

Utilizando as métricas de perfomance

 
- Coeficiente de Determinação (R2): Para medir a proporção da variância explicada. 
- Erro Médio Absoluto (MAE): Para entender o erro médio em unidades reais. 
- Erro Percentual Absoluto Médio (MAPE): Para comunicar o erro em termos percentuais (ótimo para apresentações executivas). 
- RMSE (Root Mean Squared Error): Para penalizar erros de grande magnitude. 

## 4. Veredito Final: Critérios para a Escolha do Melhor Algoritmo

A escolha do **Random Forest** como o modelo definitivo para o mercado imobiliário de Chicago baseia-se em uma análise técnica que vai além do $R^2$ elevado, atendendo aos seguintes critérios:

### 1. Análise de Resíduos e "Real vs Predito"
Ao observar o painel de **Valores Reais vs. Preditos**, o Random Forest foi o algoritmo que manteve os pontos com menor dispersão em relação à linha ideal de 45°.

* **Justificativa:** Diferente da Regressão Linear, que apresentou um padrão de erro crescente (heterocedasticidade) em casas de alto valor, o Random Forest distribuiu os resíduos de forma mais aleatória e próxima do zero. Isso indica que o modelo capturou com sucesso as **não-linearidades** dos dados de Chicago, não deixando informações estruturais "escaparem" para o erro.

### 2. Controle de Overfitting (Sobreajuste)
Monitoramos a diferença de performance entre a base de treino e a base de teste para garantir a confiabilidade.

* **Justificativa:** Embora o Random Forest seja um modelo de alta complexidade, a diferença entre os erros foi mínima. Isso prova que o modelo não apenas "decorou" os dados de treino, mas desenvolveu uma forte capacidade de **generalização**, estando pronto para prever preços de novos imóveis que não estavam no dataset original.

### 3. Interpretabilidade vs. Precisão
No setor imobiliário, a precisão financeira é o fator crítico para evitar prejuízos em avaliações de mercado.

* **Justificativa:** A Regressão Linear é mais simples de interpretar, porém, o Random Forest oferece uma precisão significativamente superior (**MAPE de 8.5%** contra **12.9%** da Linear). Em um cenário de negócios, a redução do erro percentual justifica a escolha de um modelo mais robusto. Além disso, a interpretabilidade é mantida através da análise de importância das variáveis (*Feature Importance*), que confirmou o impacto de `Space` e `Tax` no preço final.

---

### **Veredito Final**
O **Random Forest** é o modelo escolhido. Ele provou ser o mais robusto contra os **outliers** de Chicago e o mais equilibrado entre complexidade arquitetural e poder preditivo. O argumento visual da linha de 45° e a distribuição aleatória dos resíduos são as evidências definitivas de sua superioridade técnica neste projeto.

In [ ]:
# --- PAINEL COMPARATIVO: REAL VS PREDITO ---
plt.figure(figsize=(18, 5))

for i, (name, model) in enumerate(models.items()):
    # Define qual dado usar (Escalonado para Linear/SVR, Original para RF)
    xt, xv = (X_train_sc, X_test_sc) if name != "Random Forest" else (X_train, X_test)
    preds = model.predict(xv)
    
    # Criando os sub-gráficos
    plt.subplot(1, 3, i+1)
    plt.scatter(y_test, preds, alpha=0.5, color='teal')
    
    # Linha de 45 graus
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
    
    plt.title(f'{name}\nReal vs Predito')
    plt.xlabel('Preço Real')
    plt.ylabel('Preço Predito (IA)')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()